# 1. Get the data (simulate)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


data = torch.randint(0, 50257, (4, 10))

print(data)

tensor([[21050,  3813, 11573, 20545, 26399, 14595,  9399, 35321, 49199, 35664],
        [26306, 16752, 15648, 17812, 23644,  6848, 23413, 28104, 37777, 43062],
        [  199, 26504, 33532, 26162, 12100, 38767, 41039,  6633,  9723, 42511],
        [37556, 19899, 42487, 31974, 41905, 13449, 25750, 27378, 31207, 26875]])


# 2. Embedding

In [3]:
# Define the config for the model
vocab_size = 50257
d_model = 384
block_size = 1024
seq_len = 10

# Create the embedding matrixs
embedding = nn.Embedding(vocab_size, d_model)
pos_embedding = nn.Embedding(block_size, d_model)

# Embedding
embedding1 = embedding(data)
embedding2 = pos_embedding(torch.arange(seq_len))
embedding_final = embedding1 + embedding2

# Final result
print(embedding1.shape, embedding2.shape, embedding_final.shape)


torch.Size([4, 10, 384]) torch.Size([10, 384]) torch.Size([4, 10, 384])


# 3. QKV Matriz

In [4]:
Wq = nn.Linear(d_model, d_model)
Wk = nn.Linear(d_model, d_model)
Wv = nn.Linear(d_model, d_model)

Q = Wq(embedding_final)
K = Wk(embedding_final)
V = Wv(embedding_final)

print(f"Q: {Q.shape}")
print(f"K: {Q.shape}")
print(f"V: {Q.shape}")




Q: torch.Size([4, 10, 384])
K: torch.Size([4, 10, 384])
V: torch.Size([4, 10, 384])


# 4. Mask

## 1. Why the mask must have shape `(T, T)`

In scaled dot-product attention, the first step is:

```python
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
```

If `Q` has shape `[T, d_k]` and `K` has shape `[T, d_k]`, then:

```
Q @ K.T  →  [T, d_k] @ [d_k, T]  →  [T, T]
```

The result is a **T × T** matrix, where `T` is the sequence length (number of tokens). Each cell `scores[i][j]` represents **how much the query at position `i` relates to the key at position `j`**.

```
              K0     K1     K2     K3
Q0 (tok 0)   s00    s01    s02    s03
Q1 (tok 1)   s10    s11    s12    s13
Q2 (tok 2)   s20    s21    s22    s23
Q3 (tok 3)   s30    s31    s32    s33
```

Since the mask is going to be applied directly on top of `scores`, **it must have exactly the same shape**: `(T, T)`. This makes sense because the mask is a cell-by-cell map of "who is allowed to look at whom," and that map only exists in the query-key relationship — i.e., in the scores matrix itself.

```python
T = scores.shape[-1]  # sequence length
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
```


## 2. `torch.tril` vs `torch.triu`

Both are functions that **zero out one half of a matrix**, splitting it along the main diagonal. The difference is which half each one keeps.

### `torch.tril` (lower triangular)

Keeps the main diagonal **and everything below it**. Zeroes out everything above it.

```python
torch.tril(torch.ones(4, 4))
```

```
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

Interpretation in attention: row `i` (the query at position `i`) only keeps a value of `1` in columns `j <= i`. In other words, **a query can only see keys at the same position or earlier** — exactly the causal behavior we want (no peeking into the future).

### `torch.triu` (upper triangular)

Keeps the main diagonal **and everything above it**. Zeroes out everything below it.

```python
torch.triu(torch.ones(4, 4))
```

```
1 1 1 1
0 1 1 1
0 0 1 1
0 0 0 1
```

This is the opposite: each position would only see the future, never the past — not what we use for causal masking, but useful, for instance, to generate the "inverted" version of a mask or in other algorithms that need the upper half of a matrix.

### Summary of the difference

| Function | What it keeps | What it zeroes out | Use in causal attention |
|---|---|---|---|
| `torch.tril` | diagonal + below | above the diagonal | ✅ what we use (past is visible) |
| `torch.triu` | diagonal + above | below the diagonal | ❌ would represent "only seeing the future" |

In both cases, if you pass `dtype=torch.bool`, the result comes out as `True`/`False` instead of `1`/`0` — but the zero/keep logic is identical, only the representation changes.

```python
causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
```

## 3. Applying the mask with `masked_fill`

`masked_fill(mask, value)` works like an "if/else" applied cell by cell:

```
where mask[i][j] == True  → replace with `value`
where mask[i][j] == False → keep the original value
```

The catch: our `causal_mask` has `True` in the **allowed** positions (past/present), but `masked_fill` replaces exactly where it receives `True`. If we applied it directly, we'd be blocking the past and leaving the future open — the opposite of what we want.

That's why we invert the mask before applying it:

```python
causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
# True = allowed

scores = scores.masked_fill(~causal_mask, float('-inf'))
# ~causal_mask → True = blocked
# masked_fill applies -inf where it receives True → lands exactly on what was blocked
```

`~` is the bitwise NOT operator. For boolean tensors, `~causal_mask` is equivalent to `causal_mask == False` — both produce the inverted mask.

## 4. Why `-inf` and not `0`

After masking, the next step is **softmax**, applied row by row:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

If we filled the blocked positions with `0` instead of `-inf`, the token would still receive a **non-zero** probability after softmax, because:

$$e^0 = 1$$

In other words, the blocked position would still contribute (with a small but nonzero weight) to the final result — a leak of future information, even if small.

With `-inf`:

$$e^{-\infty} = 0$$

The blocked position receives **exactly zero** weight after softmax. It stops contributing at all to the final `attn_weights @ V`. This is the only way to guarantee a *complete* block, not just an "attenuated" one.

### Comparing the two cases

```python
scores_with_zero = torch.tensor([2.0, 0.0])       # "weak" block
scores_with_inf  = torch.tensor([2.0, float('-inf')])  # real block

torch.softmax(scores_with_zero, dim=0)
# tensor([0.8808, 0.1192])   ← still leaks ~12% of attention

torch.softmax(scores_with_inf, dim=0)
# tensor([1.0000, 0.0000])   ← absolute zero, no leakage
```


**One-line summary:** the mask has shape `(T, T)` because it mirrors the scores matrix produced by `Q @ K.T`; `tril` keeps the lower triangle (past visible), `triu` would do the opposite; `masked_fill` replaces values where the (inverted) mask is `True`; and we use `-inf` instead of `0` because only `-inf` guarantees an **exactly zero** weight after softmax, eliminating any leakage of future information. 

In [28]:
ones = torch.ones(4, 4,dtype=bool) # The matrix must have the shape (T, T)
mask = torch.tril(ones)
scores = torch.randn((4, 4))
scores = scores.masked_fill(mask == False , float('-inf')) # Or we can use the ~mask
print(ones)
print(mask)
print(scores)

tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])
tensor([[ 0.4833,    -inf,    -inf,    -inf],
        [-1.0084, -0.0633,    -inf,    -inf],
        [-0.2471,  0.5808,  0.3708,    -inf],
        [-2.1342,  0.8024,  1.1455, -0.1357]])


# 5. Attention

In [27]:
import torch.nn.functional as F

scores_ = F.scaled_dot_product_attention(query=Q, 
                                        key=K, 
                                        value=V,
                                        # attn_mask=mask,
                                        is_causal=True,
                                        dropout_p=0.1
                                        )
print(scores_.shape)


torch.Size([4, 10, 384])


> After seeing this function that makes all, i'll make an softmax and output part, cuz it's important 

# 6. Softmax

## The common confusion

It's natural to think of `dim` as *"the direction I want to sum/normalize, looking at the matrix."* But that's backwards from how PyTorch actually treats it.

**The real meaning:** `dim` is **the axis that gets collapsed** — the axis whose elements are combined together to produce each output value. It is *not* the axis you want to preserve; it's the axis that disappears.

```python
result = tensor.sum(dim=X)
# The values along axis X get combined into a single number.
# The remaining axes are what's left in the output shape.
```

---

## A concrete example

```python
scores = torch.tensor([
    [1.0, 2.0, 3.0],   # row 0
    [4.0, 5.0, 6.0],   # row 1
    [7.0, 8.0, 9.0]    # row 2
])
```

Shape: `[3, 3]` → axis `0` = rows, axis `1` (or `-1`) = columns.

Suppose you want **each row to sum to 1** (this is exactly what happens in attention: each query normalizes its own distribution over the keys).

To compute row 0's sum (`1.0 + 2.0 + 3.0`), you need to **walk across the 3 values of that row while keeping the row fixed** — and "walking across a row" means moving along the **column axis**. That's why the argument you pass is `dim=-1` (columns), even though intuitively you're "summing a row."

```python
scores.sum(dim=-1)
# tensor([6.0, 15.0, 24.0])
# → 3 results, one per ROW (rows survive, columns are collapsed)
```

The axis you pass (`dim`) is the one that gets combined/collapsed. The axis that remains is the one you were trying to preserve.

---

## Side-by-side comparison

```python
scores.sum(dim=-1)  # walks across columns → one sum PER ROW    → result shape: [3]
scores.sum(dim=0)   # walks across rows    → one sum PER COLUMN → result shape: [3]
```

```
sum(dim=-1):                 sum(dim=0):
[1+2+3]  = 6                 [1+4+7]  [2+5+8]  [3+6+9]
[4+5+6]  = 15                  = 12      = 15     = 18
[7+8+9]  = 24
```

---

## The quick rule of thumb

> **Want softmax across the rows?** → pick the **column** axis (`dim=-1` in a 2D `[rows, cols]` tensor).
> **Want softmax across the columns?** → pick the **row** axis (`dim=0`).

Put differently:

| You want... | You pass... | Why |
|---|---|---|
| Softmax **within each row** (each row sums to 1) | `dim=1` / `dim=-1` (the **column** axis) | The values inside a row are spread across columns — that's the axis being walked/collapsed |
| Softmax **within each column** (each column sums to 1) | `dim=0` (the **row** axis) | The values inside a column are spread across rows — that's the axis being walked/collapsed |

It feels counterintuitive at first because you're naming the axis you're *stepping through*, not the axis you're *keeping*. Once you internalize "`dim` = the axis that vanishes," it clicks.

---

## Applying this to attention

```python
scores.shape  # [seq_len_q, seq_len_k]
```

- Axis `0` (rows) = **queries** (the position that's "asking")
- Axis `-1` / `1` (columns) = **keys** (the positions that can be "answered")

We want each **query** (each row) to distribute its attention across all the **keys** it can see, so that each row sums to 1:

```python
attn_weights = torch.softmax(scores, dim=-1)
attn_weights.sum(dim=-1)  # tensor([1.0, 1.0, ...]) → every row sums to 1
```

If you accidentally used `dim=0`, you'd instead be normalizing "how much each query contributes to a fixed key" — a completely different (and meaningless, in this context) quantity, since each query is supposed to make its own independent decision about where to look.

---

## Generalizing to real attention tensors

In practice `scores` is not just `[T, T]` — it usually has batch and head dimensions too:

```python
scores.shape  # [batch, heads, seq_len_q, seq_len_k]
```

`dim=-1` still works correctly here, because it always points to the **last** dimension regardless of how many leading dimensions (`batch`, `heads`) exist. The rule never changes: you always want to normalize along the **keys** axis, which is conventionally the last dimension.

```python
attn_weights = torch.softmax(scores, dim=-1)
# shape stays [batch, heads, seq_len_q, seq_len_k]
# but now every [b, h, i, :] slice sums to 1.0
```

---

## One-line summary

**`dim` is the axis that gets combined/collapsed to produce each output value — not the axis you want to keep.** In attention, since each row (query) needs its values normalized across the columns (keys), the correct axis to pass is the column axis: `dim=-1`.

In [31]:
scores_softmax = F.softmax(scores, dim=-1)
print(scores_softmax)
print(scores.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.2799, 0.7201, 0.0000, 0.0000],
        [0.1944, 0.4449, 0.3606, 0.0000],
        [0.0186, 0.3504, 0.4939, 0.1371]])
torch.Size([4, 4])


# 7. Output

In [34]:
V = torch.rand(4, 4) # the shape of V it's (seq_len, d_model) and how we have the (T, T) after the softmax in attn we can dot them
output = scores_softmax @ V

print(f'Scores result:\n{scores_softmax}')
print(f'Output result:\n{output}')

Scores result:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.2799, 0.7201, 0.0000, 0.0000],
        [0.1944, 0.4449, 0.3606, 0.0000],
        [0.0186, 0.3504, 0.4939, 0.1371]])
Output result:
tensor([[0.8066, 0.6574, 0.7097, 0.6223],
        [0.9038, 0.4097, 0.6067, 0.1939],
        [0.6189, 0.2970, 0.6797, 0.2724],
        [0.4671, 0.2558, 0.6549, 0.3095]])
